# Test du DataLoader Keras - Cityscapes

Ce notebook teste le data loader Keras pour le dataset Cityscapes.

**Fonctionnalités testées** :
- Chargement des images et masques
- Conversion labelIds → 8 catégories
- Redimensionnement et normalisation
- One-hot encoding
- Data augmentation
- Calcul des poids de classe

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from data_loader import (
    CityscapesSequence,
    create_dataloaders,
    decode_predictions,
    colorize_mask,
    calculate_class_weights,
    CATEGORY_NAMES
)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

## 1. Créer les générateurs de données

In [ ]:
# Paramètres
DATA_ROOT = '../data'
BATCH_SIZE = 8
IMG_SIZE = (256, 512)  # (height, width)

# Créer les générateurs
train_gen, test_gen = create_dataloaders(
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    img_size=IMG_SIZE,
    augmentation=True  # Data augmentation sur train
)

print(f"\n📊 Statistiques :")
print(f"   Train : {len(train_gen.label_files)} images → {len(train_gen)} batchs")
print(f"   Test  : {len(test_gen.label_files)} images → {len(test_gen)} batchs")

## 2. Charger et visualiser un batch

In [ ]:
# Charger un batch
batch_images, batch_masks = train_gen[0]

print(f"✅ Batch chargé :")
print(f"   Images : shape={batch_images.shape}, dtype={batch_images.dtype}")
print(f"   Min={batch_images.min():.3f}, Max={batch_images.max():.3f}")
print(f"   Masques : shape={batch_masks.shape}, dtype={batch_masks.dtype}")
print(f"   Somme masque[0] = {batch_masks[0].sum():.0f} (doit être ~{IMG_SIZE[0] * IMG_SIZE[1]})")

In [ ]:
# Visualiser les 4 premières images du batch
fig, axes = plt.subplots(4, 3, figsize=(15, 16))

for i in range(4):
    # Image RGB
    axes[i, 0].imshow(batch_images[i])
    axes[i, 0].set_title(f'Image {i+1} - RGB')
    axes[i, 0].axis('off')
    
    # Masque décodé (argmax)
    mask_decoded = decode_predictions(batch_masks[i])
    axes[i, 1].imshow(mask_decoded, cmap='tab10', vmin=0, vmax=7)
    axes[i, 1].set_title(f'Masque {i+1} - Catégories (0-7)')
    axes[i, 1].axis('off')
    
    # Masque colorisé
    mask_colored = colorize_mask(mask_decoded)
    axes[i, 2].imshow(mask_colored)
    axes[i, 2].set_title(f'Masque {i+1} - Colorisé')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

print("\n✅ Visualisation OK")

## 3. Vérifier la distribution des classes dans un batch

In [ ]:
# Compter les pixels par classe dans le premier masque
mask_decoded = decode_predictions(batch_masks[0])
total_pixels = mask_decoded.size

print("Distribution des classes dans l'image 0 :")
print("-" * 60)
for cat_id in range(8):
    count = np.sum(mask_decoded == cat_id)
    percentage = (count / total_pixels) * 100
    print(f"  {cat_id} - {CATEGORY_NAMES[cat_id]:15s} : {count:6d} pixels ({percentage:5.2f}%)")
print("-" * 60)

## 4. Tester le one-hot encoding

In [ ]:
# Vérifier que le one-hot encoding est correct
mask_onehot = batch_masks[0]  # Shape: (256, 512, 8)
mask_decoded = decode_predictions(mask_onehot)  # Shape: (256, 512)

print("\nVérification du one-hot encoding :")
print(f"  Masque one-hot : shape={mask_onehot.shape}")
print(f"  Masque décodé  : shape={mask_decoded.shape}")
print(f"  Valeurs uniques dans mask décodé : {np.unique(mask_decoded)}")

# Vérifier qu'on peut retrouver le masque original
mask_reencoded = tf.keras.utils.to_categorical(mask_decoded, num_classes=8)
print(f"\n  Différence entre masque original et ré-encodé : {np.abs(mask_onehot - mask_reencoded).sum():.6f}")
print("  ✅ One-hot encoding correct !" if np.allclose(mask_onehot, mask_reencoded) else "  ❌ Problème !")

## 5. Visualiser chaque classe séparément

In [ ]:
# Visualiser les 8 canaux du masque one-hot
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for cat_id in range(8):
    axes[cat_id].imshow(batch_masks[0, :, :, cat_id], cmap='gray')
    axes[cat_id].set_title(f'{cat_id} - {CATEGORY_NAMES[cat_id]}')
    axes[cat_id].axis('off')

plt.suptitle('Masque one-hot - 8 canaux (classe par classe)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Tester la data augmentation

In [ ]:
# Comparer train (avec augmentation) et test (sans augmentation)
train_batch_img, train_batch_mask = train_gen[0]
test_batch_img, test_batch_mask = test_gen[0]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Train (avec augmentation)
for i in range(2):
    axes[0, i*2].imshow(train_batch_img[i])
    axes[0, i*2].set_title(f'Train {i+1} - Image')
    axes[0, i*2].axis('off')
    
    mask_colored = colorize_mask(decode_predictions(train_batch_mask[i]))
    axes[0, i*2+1].imshow(mask_colored)
    axes[0, i*2+1].set_title(f'Train {i+1} - Mask')
    axes[0, i*2+1].axis('off')

# Test (sans augmentation)
for i in range(2):
    axes[1, i*2].imshow(test_batch_img[i])
    axes[1, i*2].set_title(f'Test {i+1} - Image')
    axes[1, i*2].axis('off')
    
    mask_colored = colorize_mask(decode_predictions(test_batch_mask[i]))
    axes[1, i*2+1].imshow(mask_colored)
    axes[1, i*2+1].set_title(f'Test {i+1} - Mask')
    axes[1, i*2+1].axis('off')

plt.suptitle('Train (avec augmentation) vs Test (sans augmentation)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Calculer les poids de classe (pour loss pondérée)

⚠️ **Attention** : Cette opération prend ~2-3 minutes (parcourt tout le dataset train)

In [ ]:
# Calculer les poids de classe sur un sous-ensemble (pour aller plus vite)
# Pour le calcul complet, retirer le paramètre ou augmenter la valeur
print("⚠️ Calcul des poids de classe (peut prendre quelques minutes)...\n")

class_weights = calculate_class_weights(
    data_root=DATA_ROOT,
    split='train'
)

print(f"\n✅ Poids de classe calculés :")
print(f"   {class_weights}")
print(f"\n💡 Utilisation : ajouter sample_weight_mode='temporal' dans model.compile()")
print(f"   et passer class_weight={{i: w for i, w in enumerate(class_weights)}} à model.fit()")

## 8. Visualiser la distribution des poids

In [ ]:
# Graphique des poids de classe
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(CATEGORY_NAMES))
bars = ax.bar(x, class_weights, color='steelblue', alpha=0.7)

# Colorier les barres selon l'importance
for i, bar in enumerate(bars):
    if class_weights[i] > 1.5:
        bar.set_color('crimson')  # Classes rares (poids élevé)
    elif class_weights[i] < 0.5:
        bar.set_color('forestgreen')  # Classes fréquentes (poids faible)

ax.set_xticks(x)
ax.set_xticklabels(CATEGORY_NAMES, rotation=45, ha='right')
ax.set_ylabel('Poids de classe', fontsize=12)
ax.set_title('Distribution des poids de classe (inverse de la fréquence)', fontsize=14, fontweight='bold')
ax.axhline(y=1.0, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Référence (poids=1)')
ax.grid(axis='y', alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print("\n📊 Interprétation :")
print("   - Poids > 1 : Classes rares (à privilégier dans la loss)")
print("   - Poids < 1 : Classes fréquentes (à moins pénaliser)")
print("   - Rouge : Classes très rares (ex: human, vehicle)")
print("   - Vert : Classes très fréquentes (ex: flat, construction)")

## 9. Test d'intégration avec Keras

In [ ]:
# Créer un modèle dummy pour tester l'intégration
from tensorflow.keras import layers, models

def create_dummy_model(img_size=(256, 512), num_classes=8):
    """Modèle dummy pour tester le générateur"""
    inputs = layers.Input(shape=(img_size[0], img_size[1], 3))
    
    # Architecture très simple (juste pour tester)
    x = layers.Conv2D(16, 3, padding='same', activation='relu')(inputs)
    x = layers.Conv2D(num_classes, 1, padding='same', activation='softmax')(x)
    
    model = models.Model(inputs, x)
    return model

# Créer le modèle
dummy_model = create_dummy_model(img_size=IMG_SIZE)
dummy_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n✅ Modèle dummy créé :")
dummy_model.summary()

# Tester un fit sur 1 batch
print("\n🔄 Test d'entraînement sur 1 batch...")
history = dummy_model.fit(
    train_gen,
    steps_per_epoch=1,  # 1 seul batch
    epochs=1,
    verbose=1
)

print("\n✅ Test d'intégration avec Keras OK !")

## 10. Test de prédiction

In [ ]:
# Faire une prédiction avec le modèle dummy
test_batch_img, test_batch_mask = test_gen[0]
predictions = dummy_model.predict(test_batch_img[:2])  # Prédire sur 2 images

print(f"\n✅ Prédictions :")
print(f"   Shape : {predictions.shape}")
print(f"   Min : {predictions.min():.4f}, Max : {predictions.max():.4f}")

# Visualiser
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i in range(2):
    # Image
    axes[i, 0].imshow(test_batch_img[i])
    axes[i, 0].set_title(f'Image {i+1}')
    axes[i, 0].axis('off')
    
    # Ground truth
    gt_mask = colorize_mask(decode_predictions(test_batch_mask[i]))
    axes[i, 1].imshow(gt_mask)
    axes[i, 1].set_title(f'Ground Truth {i+1}')
    axes[i, 1].axis('off')
    
    # Prédiction
    pred_mask = colorize_mask(decode_predictions(predictions[i]))
    axes[i, 2].imshow(pred_mask)
    axes[i, 2].set_title(f'Prédiction {i+1} (dummy model)')
    axes[i, 2].axis('off')

plt.suptitle('Image | Ground Truth | Prédiction (modèle non entraîné)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✅ Toutes les fonctionnalités testées avec succès !")

## 📝 Résumé

✅ **DataLoader Keras opérationnel** :
- Charge images + masques Cityscapes
- Convertit labelIds (0-33) → 8 catégories (0-7)
- Redimensionne à la taille voulue
- Normalise les pixels (0-1)
- One-hot encoding des masques
- Data augmentation (train)
- Calcul des poids de classe
- Compatible avec Keras (keras.utils.Sequence)

**Prêt pour l'entraînement du modèle !** 🚀